<a href="https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/NB7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TomazDrumond/Tom_Fly/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [4]:
import os, sys, pandas as pd, numpy as np, duckdb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

rel = "hf://datasets/FlyRank/internship-warehouse"
df_content = pd.read_parquet(f"{rel}/dim_content.parquet", storage_options={"token": hf_token})
df_march = pd.read_parquet(f"{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet", storage_options={"token": hf_token})
df_april = pd.read_parquet(f"{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet", storage_options={"token": hf_token})

con = duckdb.connect()
con.register("march", df_march)
con.register("april", df_april)

def month_agg(table):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
               AVG(gsc_avg_position) AS avg_position
        FROM {table} WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    """).df()

march_agg, april_agg = month_agg("march"), month_agg("april")
march_agg["ctr"] = march_agg["clicks"] / march_agg["impressions"].replace(0, np.nan)
april_agg["ctr"] = april_agg["clicks"] / april_agg["impressions"].replace(0, np.nan)

panel = march_agg.merge(april_agg, on=["client_hash_id", "content_hash_id"], suffixes=("_march", "_april"))
panel = panel[(panel["impressions_march"] >= 100) & (panel["impressions_april"] > 0)].copy()
panel["is_declining_label"] = (panel["ctr_april"] < panel["ctr_march"]).astype(int)
panel = panel.merge(df_content[["client_hash_id", "content_hash_id", "word_count"]],
                     on=["client_hash_id", "content_hash_id"], how="left")

feature_cols = ["impressions_march", "avg_position_march", "ctr_march", "word_count"]
X = panel[feature_cols].fillna(0)
y = panel["is_declining_label"].values
groups = panel["client_hash_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
model = LogisticRegression(max_iter=1000, random_state=42).fit(X.iloc[train_idx], y[train_idx])

print("Model retrained standalone. Panel shape:", panel.shape)

Model retrained standalone. Panel shape: (100893, 12)


In [5]:
# Score the FULL panel with the honest, grouped-split model — this is the actual playbook,
# meant to cover everyone, not just held-out evaluation rows
full_scores = model.predict_proba(panel[feature_cols].fillna(0))[:, 1]
panel["risk_score"] = full_scores

# Reason codes: WHY the model flagged this page, in plain terms a reviewer trusts
def assign_reason(row):
    if row["ctr_march"] > panel["ctr_march"].quantile(0.75) and row["avg_position_march"] <= 10:
        return "strong_march_ctr_at_risk_of_reverting"
    elif row["avg_position_march"] > 20:
        return "poor_position_visibility_risk"
    elif row["impressions_march"] < 500:
        return "low_volume_low_confidence"
    else:
        return "elevated_decline_risk"

panel["reason_code"] = panel.apply(assign_reason, axis=1)

def assign_action(row):
    if row["risk_score"] >= 0.7 and row["avg_position_march"] <= 10:
        return "review_for_refresh"
    elif row["risk_score"] >= 0.7:
        return "monitor_closely"
    elif row["risk_score"] < 0.3:
        return "protect_as_is"
    else:
        return "routine_monitor"

panel["action"] = panel.apply(assign_action, axis=1)

queue = panel.sort_values("risk_score", ascending=False).reset_index(drop=True)

print("Queue size:", len(queue))
print("\nAction distribution:")
print(queue["action"].value_counts())
print("\nReason code distribution:")
print(queue["reason_code"].value_counts())
queue[["client_hash_id", "content_hash_id", "risk_score", "reason_code", "action",
       "impressions_march", "avg_position_march", "ctr_march"]].head(10)

Queue size: 100893

Action distribution:
action
routine_monitor       83430
protect_as_is         14507
review_for_refresh     2609
monitor_closely         347
Name: count, dtype: int64

Reason code distribution:
reason_code
elevated_decline_risk                    38218
poor_position_visibility_risk            23962
low_volume_low_confidence                21121
strong_march_ctr_at_risk_of_reverting    17592
Name: count, dtype: int64


,client_hash_id,content_hash_id,risk_score,reason_code,action,impressions_march,avg_position_march,ctr_march
0,client_e547b89c05043229,content_eadb33b5df496f4a,1.000000,strong_march_ctr_at_risk_of_reverting,review_for_refresh,617124.0,2.383011,0.009185
1,client_e547b89c05043229,content_ec2e0346994fb5a5,0.999995,strong_march_ctr_at_risk_of_reverting,review_for_refresh,245276.0,2.854514,0.006034
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,0.999991,elevated_decline_risk,monitor_closely,244931.0,15.008339,0.002731
3,client_e547b89c05043229,content_0e03de7680314cd5,0.999982,elevated_decline_risk,review_for_refresh,221310.0,2.675217,0.003253
4,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,0.999971,strong_march_ctr_at_risk_of_reverting,review_for_refresh,205045.0,4.544203,0.011929
5,client_23a62021009f63c4,content_44f34c0a90047651,0.999964,elevated_decline_risk,review_for_refresh,212404.0,7.346909,0.000113
6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,0.999962,strong_march_ctr_at_risk_of_reverting,review_for_refresh,205867.0,3.367835,0.004187
7,client_e547b89c05043229,content_8d7d99f109e19aa2,0.999953,elevated_decline_risk,review_for_refresh,203497.0,2.563756,0.001420
8,client_62f4a7e64f5e0096,content_f107e54b10b43725,0.999928,strong_march_ctr_at_risk_of_reverting,review_for_refresh,195997.0,3.186054,0.005082
9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,0.999922,elevated_decline_risk,review_for_refresh,194337.0,4.450106,0.001858


The queue ranks all 100,893 pages in the panel by predicted decline risk, using the honest
grouped-split Logistic Regression model (Week-5/Week-6) scored on March-only features. Each
row carries a reason code explaining *why* it was flagged:

- `elevated_decline_risk` (38,218 pages, 37.9%) — general risk signal, no single dominant trait
- `poor_position_visibility_risk` (23,962, 23.8%) — already ranking poorly (position 20+)
- `low_volume_low_confidence` (21,121, 20.9%) — low March impressions, treat scores here cautiously
- `strong_march_ctr_at_risk_of_reverting` (17,592, 17.4%) — the ceiling-effect pattern
  identified in Week-5's error analysis: already-high CTR pages the model considers likely
  to regress toward the mean

Actions split the queue by urgency and confidence: `review_for_refresh` (2,609 pages, 2.6%)
is the actual actionable review list; `monitor_closely` (347, 0.3%) flags high-risk pages
outside the top-10 position band, where the model's position-80+ blind spot (Week-6) makes
a "review" action premature without a manual visibility check first; `protect_as_is`
(14,507, 14.4%) and `routine_monitor` (83,430, 82.7%) require no near-term action. Several
of the highest-risk pages in this run match pages that also scored highly in the Week-4
hand-written baseline — a useful cross-check that the two independent approaches agree on
at least some real signal, not just coincidence.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [6]:
print("Total distinct clients in queue:", panel["client_hash_id"].nunique())
print("Clients with at least 1 review_for_refresh page:",
      queue[queue["action"] == "review_for_refresh"]["client_hash_id"].nunique())

Total distinct clients in queue: 43
Clients with at least 1 review_for_refresh page: 29


**Intended use:** a content reviewer or SEO strategist working through a limited-capacity
review cycle, using this queue to decide which pages to look at first this cycle — not to
automatically apply changes. 29 of the panel's 43 clients (67%) have at least one page
flagged `review_for_refresh`; a reviewer working across multiple client accounts can use
this queue as a starting triage list per client, not just a single flat ranking.

**Where it stops being valid:**
- The 14 clients (33%) with zero flagged pages should not be read as "confirmed healthy" —
  it may equally mean their pages didn't clear the volume/position thresholds behind the
  reason codes, or that their GSC coverage in this panel is too thin to produce a confident
  score (see Week-3's per-client availability finding: median client availability was only
  8%). Absence of a flag is not evidence of absence of a problem.
- Evaluated on a single March→April transition; performance on other month-pairs, seasons,
  or after a major algorithm change is untested.
- The label (CTR decline) is a proxy for "needs a refresh," not a measurement of whether
  refreshing actually helps — this queue prioritizes review, it does not promise recovery.
- Known model blind spot (Week-6's error audit): the model under-detects decline in already
  low-ranked pages (position 80+) that lose visibility entirely rather than merely
  converting worse — a low `risk_score` on a poorly-ranked page should not be read as
  reassurance, which is exactly why those cases route to `monitor_closely` rather than
  `protect_as_is` in this queue's action logic.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [7]:
# How many "review_for_refresh" pages are ALSO in the risky low-confidence bucket?
overlap = queue[(queue["action"] == "review_for_refresh") & (queue["reason_code"] == "low_volume_low_confidence")]
print("review_for_refresh pages with low_volume_low_confidence reason code:", len(overlap))


review_for_refresh pages with low_volume_low_confidence reason code: 0


**Sanity check confirmed:** 0 pages carry both `review_for_refresh` (highest urgency action)
and `low_volume_low_confidence` (lowest confidence reason code) — the action logic doesn't
contradict itself by urgently flagging a page it simultaneously admits it isn't confident
about. This is a design property worth stating explicitly rather than assuming.

**Before acting on any flagged page, a human must check:**
- Whether the page's content actually still matches current search intent for its target
  query — the model has no signal for topical drift.
- Whether a recent site change (URL migration, redirect, tracking break) explains a score,
  rather than genuine content decline — Week-4's top-20 review found this exact
  false-positive pattern on real pages.
- Whether the page is intentionally low-CTR by design (e.g., informational content where
  users get their answer from the search snippet) before flagging it for refresh — Week-5's
  error analysis showed the model can't distinguish this from genuine underperformance.
- For any `monitor_closely` page (347 total): manually confirm current visibility status,
  since these exist specifically because the model's position-80+ blind spot makes an
  automatic "review" action premature without a human visibility check first.

**What should NEVER be automated:**
- Auto-publishing or auto-editing any page content based on `risk_score` alone.
- Auto-deprioritizing or pruning a page purely because of `protect_as_is` status (14,507
  pages) — this label means "no signal of decline in this model," not "verified healthy,"
  especially for the 14 of 43 clients with no flagged pages at all (Section 2).
- Treating the ranked order as a guarantee of impact — the queue orders review priority,
  not expected ROI per page.
- Applying this playbook's scores at face value for clients with very sparse March GSC
  coverage (Week-3 finding: 8 of 55 clients had 0% availability) — the model has no real
  signal for these clients regardless of what score it outputs.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [8]:
print("Reference base rate (declining):", round(panel["is_declining_label"].mean(), 3))
print("Reference median risk_score:", round(panel["risk_score"].median(), 3))


Reference base rate (declining): 0.442
Reference median risk_score: 0.457


In [9]:
import json

monitoring_reference = {
    "base_rate_declining": 0.442,
    "median_risk_score": 0.457,
    "n_clients_in_training_panel": 43,
    "clients_with_flagged_pages": 29,
    "precision_at_20_grouped_holdout": 0.55,
    "precision_at_50_grouped_holdout": 0.70
}
print(json.dumps(monitoring_reference, indent=2))

{
  "base_rate_declining": 0.442,
  "median_risk_score": 0.457,
  "n_clients_in_training_panel": 43,
  "clients_with_flagged_pages": 29,
  "precision_at_20_grouped_holdout": 0.55,
  "precision_at_50_grouped_holdout": 0.7
}


**Retrain/re-review triggers** (checked periodically by a human, not automated):
- The month-over-month base rate of declining pages shifts substantially from the
  reference (0.442) — suggests the underlying March→April pattern may no longer generalize
  to the current period.
- A fresh month's held-out Precision@20/50, re-evaluated with the same grouped-split method,
  falls meaningfully below the reference values (0.55 / 0.70).
- The median `risk_score` drifts far from the reference (0.457) without a corresponding
  shift in the actual base rate — a sign the model's calibration has degraded even if
  ranking order is still roughly sound.
- The proportion of clients with zero flagged pages (currently 14 of 43, 33%) rises
  significantly — could indicate coverage/availability issues rather than genuine health.
- More than ~3 months since the last retrain, given this was validated on a single
  month-pair transition only.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
import os

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Ranked queue — regenerated every run, stays out of git per the leak-guard
queue[["client_hash_id", "content_hash_id", "risk_score", "reason_code", "action",
       "impressions_march", "avg_position_march", "ctr_march"]].to_csv(
    "work/outputs/action_playbook_queue.csv", index=False
)

# Metrics JSON — committed; the receipt the paper's numbers trace back to
metrics = {
    "model": "Logistic Regression",
    "split_type": "grouped_by_client",
    "baseline_precision_at_20": 0.50,
    "model_precision_at_20": 0.55,
    "baseline_precision_at_50": 0.54,
    "model_precision_at_50": 0.70,
    "test_base_rate": 0.438,
    **monitoring_reference,
    "action_distribution": queue["action"].value_counts().to_dict(),
    "reason_code_distribution": queue["reason_code"].value_counts().to_dict()
}

with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported:")
print("- work/outputs/action_playbook_queue.csv (", len(queue), "rows )")
print("- work/outputs/playbook_metrics.json")
print(json.dumps(metrics, indent=2))

Exported:
- work/outputs/action_playbook_queue.csv ( 100893 rows )
- work/outputs/playbook_metrics.json
{
  "model": "Logistic Regression",
  "split_type": "grouped_by_client",
  "baseline_precision_at_20": 0.5,
  "model_precision_at_20": 0.55,
  "baseline_precision_at_50": 0.54,
  "model_precision_at_50": 0.7,
  "test_base_rate": 0.438,
  "base_rate_declining": 0.442,
  "median_risk_score": 0.457,
  "n_clients_in_training_panel": 43,
  "clients_with_flagged_pages": 29,
  "precision_at_20_grouped_holdout": 0.55,
  "precision_at_50_grouped_holdout": 0.7,
  "action_distribution": {
    "routine_monitor": 83430,
    "protect_as_is": 14507,
    "review_for_refresh": 2609,
    "monitor_closely": 347
  },
  "reason_code_distribution": {
    "elevated_decline_risk": 38218,
    "poor_position_visibility_risk": 23962,
    "low_volume_low_confidence": 21121,
    "strong_march_ctr_at_risk_of_reverting": 17592
  }
}


`action_playbook_queue.csv` regenerates on every run (not committed, per the leak-guard).
`playbook_metrics.json` is committed — it's the receipt this paper's Results and
Recommendations numbers trace back to, including the action/reason-code distributions
confirmed above. No figures were generated in this notebook; the Precision@K comparison
chart already built for the deployed paper (Week-7/8) can be regenerated from these same
metrics if a committed image is needed in `work/figures/`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.